# NetCDF vs Zarr for scalable Earth system data

This notebook compares NetCDF and Zarr formats in the context of
cloud- and HPC-friendly data access.


NetCDF:
- Optimized for sequential access
- Widely used in climate modelling
- Chunking fixed at write time

Zarr:
- Chunked, cloud-native format
- Designed for parallel access
- Well suited for Dask-based workflows

" Zarr is motivated by the need for a simple, transparent, open, and community-driven format that supports high-throughput distributed I/O on different storage systems. Zarr data can be stored in any storage system that can be represented as a key-value store, including most commonly POSIX file systems and cloud object storage but also zip files as well as relational and document databases."

In [1]:
import matplotlib.pyplot as plt # type: ignore
import numpy as np # type: ignore
import xarray as xr # type: ignore
import dask # type: ignore
import dask.array as da # type: ignore
import zarr

xr.set_options(keep_attrs=True, display_expand_data=False)
np.set_printoptions(threshold=10, edgeitems=2)

%xmode minimal
%matplotlib inline
%config InlineBackend.figure_format='retina'

Exception reporting mode: Minimal


In [2]:
from dask.distributed import Client # type: ignore

client = Client()
client


/Users/samin91/Desktop/Projects/earth-data-xarray-netcdf-zarr-demo/venv/lib/python3.9/site-packages/distributed/node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 54720 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:54720/status,
Dashboard: http://127.0.0.1:54720/status,Workers: 11
Total threads: 11,Total memory: 18.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:54721,Workers: 11
Dashboard: http://127.0.0.1:54720/status,Total threads: 11
Started: Just now,Total memory: 18.00 GiB
Comm: tcp://127.0.0.1:54755,Total threads: 1
Dashboard: http://127.0.0.1:54757/status,Memory: 1.64 GiB
Nanny: tcp://127.0.0.1:54724,


In [3]:
# open the dataset as a dask array
ds = xr.open_dataset("../dataset/era5_temp.nc", chunks={'time': 'auto'
                                                        #, 
                                                        #"latitude": 2, 
                                                        #"longitude": 2
                                                        })
ds 

<xarray.Dataset> Size: 36GB
Dimensions:     (valid_time: 8760, latitude: 721, longitude: 1440)
Coordinates:
    number      int64 8B ...
  * valid_time  (valid_time) datetime64[ns] 70kB 2025-01-01 ... 2025-12-31T23...
  * latitude    (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * longitude   (longitude) float64 12kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
    expver      (valid_time) <U4 140kB dask.array<chunksize=(8760,), meta=np.ndarray>
Data variables:
    t2m         (valid_time, latitude, longitude) float32 36GB dask.array<chunksize=(674, 52, 103), meta=np.ndarray>
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-01-06T10:57 GRIB to CDM+CF via cfgrib-0.9.1...

In [8]:
ds.t2m.chunks

((674, 674, 674, 674, 674, 674, 674, 674, 674, 674, 674, 674, 672),
 (52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 45),
 (103, 103, 103, 103, 103, 103, 103, 103, 103, 103, 103, 103, 103, 101))

In [9]:
ds.t2m.data

dask.array<open_dataset-t2m, shape=(8760, 721, 1440), dtype=float32, chunksize=(674, 52, 103), chunktype=numpy.ndarray>

#### Convert NetCDF to Zarr

In [ ]:
subset = ds.isel(valid_time=slice(0, 100))
path = '/Users/samin91/Desktop/Projects/earth-data-xarray-netcdf-zarr-demo/dataset/era5_temp_small.zarr'
if not zarr.storage.exists(path):
    subset.to_zarr(path, mode="w")


In [10]:
subset.t2m.data

dask.array<getitem, shape=(100, 721, 1440), dtype=float32, chunksize=(100, 52, 103), chunktype=numpy.ndarray>

#### Open Zarr and inspect chunks

In [5]:

ds_zarr = xr.open_zarr(path)

In [7]:
ds_zarr.t2m.chunks

((100,),
 (52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 52, 45),
 (103, 103, 103, 103, 103, 103, 103, 103, 103, 103, 103, 103, 103, 101))